In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import pickle
import matplotlib.pyplot as plt

def split_scalar(indep_X,dep_Y):
        X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
        
        sc = StandardScaler()
        X_train = sc.fit_transform(X_train)
        X_test = sc.transform(X_test)    
        return X_train, X_test, y_train, y_test
    
def r2_prediction(regressor,X_test,y_test):
     y_pred = regressor.predict(X_test)
     from sklearn.metrics import r2_score
     r2=r2_score(y_test,y_pred)
     return r2
 
def Linear(X_train,y_train,X_test,y_test):       
        from sklearn.linear_model import LinearRegression
        regressor = LinearRegression()
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2   
    
def svm_linear(X_train,y_train,X_test,y_test):
        from sklearn.svm import SVR
        regressor = SVR(kernel = 'linear')
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
    
def svm_NL(X_train,y_train,X_test,y_test):
        from sklearn.svm import SVR
        regressor = SVR(kernel = 'rbf')
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
     
def Decision(X_train,y_train,X_test,y_test):
        from sklearn.tree import DecisionTreeRegressor
        regressor = DecisionTreeRegressor(random_state = 0)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
     
def random(X_train,y_train,X_test,y_test):       
        from sklearn.ensemble import RandomForestRegressor
        regressor = RandomForestRegressor(n_estimators = 10, random_state = 0)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2 

def ldaFeature(indep_X,dep_Y,n):
        ldalist=[]
        
        lda = LinearDiscriminantAnalysis(n_components=n)
        indep_X_lda = lda.fit_transform(indep_X, dep_Y)
        ldalist.append(indep_X_lda)
        
        return ldalist
    
def lda_regression(acclog,accsvml,accdes,accrf): 
    
    ldadataframe=pd.DataFrame(index=['LDA'],columns=['Linear','SVMl','Decision','Random'])

    for number,idex in enumerate(ldadataframe.index):
        ldadataframe.loc[idex,'Linear']=acclog[number]       
        ldadataframe.loc[idex,'SVMl']=accsvml[number]
        ldadataframe.loc[idex,'Decision']=accdes[number]
        ldadataframe.loc[idex,'Random']=accrf[number]
    return ldadataframe


dataset1=pd.read_csv("prep.csv",index_col=None)
df2=dataset1
df2 = pd.get_dummies(df2, drop_first=True)

indep_X=df2.drop('classification_yes', axis=1)
dep_Y=df2['classification_yes']

# scale before LDA
sc = StandardScaler()
indep_X = sc.fit_transform(indep_X)

ldalist=ldaFeature(indep_X,dep_Y,1)       

acclin=[]
accsvml=[]
accsvmnl=[]
accdes=[]
accrf=[]

for i in ldalist:   
    X_train, X_test, y_train, y_test=train_test_split(i, dep_Y, test_size = 0.25, random_state = 0)
    
    r2_lin=Linear(X_train,y_train,X_test,y_test)
    acclin.append(r2_lin)
    
    r2_sl=svm_linear(X_train,y_train,X_test,y_test)    
    accsvml.append(r2_sl)
    
    r2_NL=svm_NL(X_train,y_train,X_test,y_test)
    accsvmnl.append(r2_NL)
    
    r2_d=Decision(X_train,y_train,X_test,y_test)
    accdes.append(r2_d)
    
    r2_r=random(X_train,y_train,X_test,y_test)
    accrf.append(r2_r)
    
result=lda_regression(acclin,accsvml,accdes,accrf)



In [2]:
result

,Linear,SVMl,Decision,Random
LDA,0.750587,0.73936,0.956597,0.977865
